In [1]:
# Importando bibliotecas do sistema para navegar nas pastas
import sys
import os

# Adicionando a pasta raiz do projeto (fincontrol) ao caminho do Python
sys.path.append(os.path.abspath('..'))

# Importando a nossa função criada no extractor.py
from src.extractor import extract_nubank_credit_card

# Definindo o caminho para o primeiro arquivo da sua lista
caminho_arquivo = '../data/bronze/credit_cards/nubank/Nubank_2017-02-11.ofx'

# Executando a extração
df_cartao = extract_nubank_credit_card(caminho_arquivo)

# Exibindo as primeiras linhas do resultado
display(df_cartao.head())

,id_transacao,data_compra,valor,descricao,tipo
0,588070fa-9ce6-47b3-84b6-ebfda3da223d,2017-01-18 03:00:00,40.16,Pagamento recebido,credit
1,5880724c-8d7c-41be-9134-a913f58e6e7d,2017-01-19 03:00:00,-36.75,Apl* Itunes.Com/Bill,debit
2,5880724c-e926-4343-9fde-c090122b008e,2017-01-19 03:00:00,-2.34,"IOF de ""Apl* Itunes.Com/Bill""",debit


In [2]:
import os
from src.transformer import clean_credit_card_data

# 1. Passando o dado bruto pela nossa função de limpeza
df_silver = clean_credit_card_data(df_cartao)

# 2. Criando o caminho da pasta Silver (garantindo que ela exista na nossa estrutura)
caminho_silver = '../data/silver/credit_cards/nubank/'
os.makedirs(caminho_silver, exist_ok=True)

# 3. Definindo o nome do arquivo de saída e salvando em formato Parquet
arquivo_saida = os.path.join(caminho_silver, 'Nubank_2017-02-11.parquet')
df_silver.to_parquet(arquivo_saida, index=False)

print(f"Arquivo salvo com sucesso em: {arquivo_saida}")

# Exibindo o resultado final limpo
display(df_silver.head())

Arquivo salvo com sucesso em: ../data/silver/credit_cards/nubank/Nubank_2017-02-11.parquet


,id_transacao,data_compra,valor,descricao,tipo
0,588070fa-9ce6-47b3-84b6-ebfda3da223d,2017-01-18 03:00:00,40.16,PAGAMENTO RECEBIDO,credit
1,5880724c-8d7c-41be-9134-a913f58e6e7d,2017-01-19 03:00:00,-36.75,APL* ITUNES.COM/BILL,debit
2,5880724c-e926-4343-9fde-c090122b008e,2017-01-19 03:00:00,-2.34,"IOF DE ""APL* ITUNES.COM/BILL""",debit


In [3]:
import os
import glob
import pandas as pd
from src.extractor import extract_nubank_credit_card
from src.transformer import clean_credit_card_data

# 1. Definir a pasta onde estão os arquivos brutos do Nubank
caminho_bronze_nubank = '../data/bronze/credit_cards/nubank/'

# O glob vai listar todos os arquivos terminados em .ofx dentro dessa pasta
arquivos_ofx = glob.glob(os.path.join(caminho_bronze_nubank, '*.ofx'))
print(f"Encontrados {len(arquivos_ofx)} arquivos OFX para processar. Iniciando...\n")

# 2. Lista vazia que vai guardar os "pedacinhos" de dados de cada arquivo
lista_dfs = []

# 3. O Loop da automação
for arquivo in arquivos_ofx:
    try:
        # Extrai (Bronze)
        df_bruto = extract_nubank_credit_card(arquivo)
        
        # Limpa e padroniza (Silver)
        df_limpo = clean_credit_card_data(df_bruto)
        
        # Guarda na lista
        lista_dfs.append(df_limpo)
        
    except Exception as e:
        # Se algum arquivo estiver corrompido, o código avisa e continua os outros
        print(f"⚠️ Erro ao processar o arquivo {os.path.basename(arquivo)}: {e}")

# 4. Juntando as peças e finalizando
if lista_dfs:
    # Empilha todos os DataFrames da lista em um só
    df_consolidado = pd.concat(lista_dfs, ignore_index=True)
    
    # REGRA DE OURO: Remover duplicadas! 
    # É comum termos arquivos de meses sobrepostos e exportar a mesma compra duas vezes.
    # Usamos o 'id_transacao' que o OFX nos fornece para garantir que cada compra seja única.
    qtd_antes = len(df_consolidado)
    df_consolidado = df_consolidado.drop_duplicates(subset=['id_transacao'])
    qtd_depois = len(df_consolidado)
    
    # Ordena pelo tempo para ficar organizado
    df_consolidado = df_consolidado.sort_values(by='data_compra').reset_index(drop=True)
    
    # 5. Salva o histórico final na camada Silver
    caminho_silver_consolidado = '../data/silver/credit_cards/nubank/nubank_historico_consolidado.parquet'
    df_consolidado.to_parquet(caminho_silver_consolidado, index=False)
    
    print(f"✅ Sucesso! O histórico final foi salvo.")
    print(f"   - Transações totais processadas: {qtd_antes}")
    print(f"   - Transações únicas consolidadas: {qtd_depois}")
    print(f"   - Foram removidas {qtd_antes - qtd_depois} compras duplicadas.\n")
    
    display(df_consolidado.head())
else:
    print("Nenhum dado foi consolidado.")

Encontrados 112 arquivos OFX para processar. Iniciando...

✅ Sucesso! O histórico final foi salvo.
   - Transações totais processadas: 6579
   - Transações únicas consolidadas: 6063
   - Foram removidas 516 compras duplicadas.



,id_transacao,data_compra,valor,descricao,tipo
0,588070fa-9ce6-47b3-84b6-ebfda3da223d,2017-01-18 03:00:00,40.16,PAGAMENTO RECEBIDO,credit
1,5880724c-8d7c-41be-9134-a913f58e6e7d,2017-01-19 03:00:00,-36.75,APL* ITUNES.COM/BILL,debit
2,5880724c-e926-4343-9fde-c090122b008e,2017-01-19 03:00:00,-2.34,"IOF DE ""APL* ITUNES.COM/BILL""",debit
3,588b071b-e876-4e70-a174-68dfbac60aa6,2017-01-27 03:00:00,-28.24,CAFFE RESTAURANTES,debit
4,588b7523-52da-481d-afe8-422b33595e66,2017-01-27 03:00:00,-27.01,AUTO POSTO AMAZONAS,debit
